# Verify a model: the trust card catching bad predictions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PINNeAPPle-Labs/PINNeAPPle/blob/main/notebooks/colab/05_verify_a_model.ipynb)

A prediction can look right, have a smooth colour map and a reasonable range, and still be wrong. This notebook takes
one problem (steady heat in a plate, $\nabla^2 T = 0$, $T = \sin(\pi x)$ on the top edge, 0 elsewhere) and four
predictions of it:

| prediction | what is wrong with it |
|---|---|
| a trained PINN | nothing (the control) |
| an under-trained PINN | it has not learned the equation yet |
| a field that obeys the equation but not the boundary | a plausible, smooth, harmonic field heated 20 % too much |
| the trained PINN queried outside its training region | extrapolation (out of distribution) |

and shows which check of the **trust card** catches each one. Use the same card on any field you get, from any code:
it needs only the values on a grid.

In [ ]:
# Setup: on Colab (or wherever PINNeAPPle is missing) install it from PyPI, then pick the device.
import importlib.util, os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB or importlib.util.find_spec("pinneapple") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pinneapple>=0.7.0"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import torch
import pinneapple as pp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PINNeAPPle {pp.__version__} on {DEVICE}")

In [ ]:
# Parameters. SMOKE=True shrinks every size so the notebook runs in seconds (the CI runs it that way).
SMOKE = os.environ.get("PINNEAPPLE_NB_SMOKE", "0") == "1"
EPOCHS_GOOD = 400 if SMOKE else 2000    # the trained model
EPOCHS_BAD = 20                         # the under-trained model
N_COLLOCATION = 1024 if SMOKE else 2048
GRID = 33 if SMOKE else 65

In [ ]:
from pathlib import Path
import urllib.request
from pinneapple_physics.pde_environment.builder import ProblemBuilder

REPO_RAW = "https://raw.githubusercontent.com/PINNeAPPle-Labs/PINNeAPPle/main/"

def repo_file(rel):
    """A file of the PINNeAPPle repository: from the local checkout if there is one, else downloaded from GitHub."""
    roots = [Path(os.environ["PINNEAPPLE_REPO"])] if os.environ.get("PINNEAPPLE_REPO") else []
    for root in roots + [Path.cwd(), *Path.cwd().parents]:
        if (root / rel).is_file() and (root / "pinneapple").is_dir():
            return root / rel
    dest = Path("pinneapple_files") / rel
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

def load_trust_card():
    """pp.trust_card (PINNeAPPle newer than 0.7.0), else the same module loaded from the repository."""
    try:
        return pp.trust_card
    except AttributeError:
        spec_ = importlib.util.spec_from_file_location("field_card", repo_file("pinneapple_analysis/trust/field_card.py"))
        mod = importlib.util.module_from_spec(spec_)
        sys.modules["field_card"] = mod
        spec_.loader.exec_module(mod)
        return mod.trust_card

trust_card = load_trust_card()

def top_edge(X):
    return np.sin(np.pi * X[:, 0])

spec = (ProblemBuilder("heated_plate").domain(x=(0, 1), y=(0, 1)).fields("T").pde("laplace")
        .bc("dirichlet", field="T", value=top_edge, on="y_max")
        .bc("dirichlet", field="T", value=0.0, on="y_min")
        .bc("dirichlet", field="T", value=0.0, on="x_boundary").build())
problem = pp.PhysicalProblem.from_pde_spec(spec)
EDGES = {"x_min": 0.0, "x_max": 0.0, "y_min": 0.0, "y_max": top_edge}

def on_grid(fn, y_max=1.0):
    x, y = np.linspace(0, 1, GRID), np.linspace(0, y_max, GRID)
    X, Y = np.meshgrid(x, y, indexing="ij")
    return {"x": x, "y": y}, fn(np.stack([X.ravel(), Y.ravel()], axis=1)).reshape(X.shape)

## Four predictions

In [ ]:
good = pp.solve(problem, "pinn", epochs=EPOCHS_GOOD, n_collocation=N_COLLOCATION, hidden_dim=48, device=DEVICE, seed=0)
rushed = pp.solve(problem, "pinn", epochs=EPOCHS_BAD, n_collocation=N_COLLOCATION, hidden_dim=48, device=DEVICE, seed=0)

def harmonic_too_hot(X):        # obeys Laplace exactly, but the top edge is at 1.2 sin(pi x)
    return 1.2 * np.sin(np.pi * X[:, 0]) * np.sinh(np.pi * X[:, 1]) / np.sinh(np.pi)

cases = {
    "trained PINN": on_grid(lambda P: good.predict(P)[:, 0]),
    "under-trained PINN": on_grid(lambda P: rushed.predict(P)[:, 0]),
    "harmonic, wrong boundary": on_grid(harmonic_too_hot),
    "trained PINN, y up to 1.5": on_grid(lambda P: good.predict(P)[:, 0], y_max=1.5),
}
fig, axs = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, (name, (grid, T)) in zip(axs, cases.items()):
    im = ax.pcolormesh(grid["x"], grid["y"], T.T, shading="auto", cmap="inferno")
    fig.colorbar(im, ax=ax)
    ax.set(title=name, xlabel="x", ylabel="y")
plt.tight_layout()
plt.show()

All four look like a heated plate. Now the checks. There is **no reference solution** here on purpose: in practice
you rarely have one, and the physics alone must catch the problems. The training bounds of the networks
($x, y \in [0, 1]$) feed the applicability check.

In [ ]:
cards = {}
for name, (grid, T) in cases.items():
    edges = dict(EDGES)
    if grid["y"][-1] > 1.0:
        edges.pop("y_max")                  # the top edge of the queried region is not the plate's top edge
    cards[name] = trust_card(grid, T, name=name, pde="laplace", dirichlet=edges,
                             train_bounds={"x": (0, 1), "y": (0, 1)})
    print(cards[name], "\n")

In [ ]:
import pandas as pd

names = list(cards)
checks = sorted({c.name for card in cards.values() for c in card.checks}, key=lambda n: n.lower())
mark = {True: "pass", False: "FAIL", None: "-"}
pd.DataFrame({n: [mark[next((c.passed for c in cards[n].checks if c.name == ck), None)] for ck in checks] +
              [cards[n].decision] for n in names}, index=checks + ["DECISION"])

## What caught what

* **Under-trained PINN**: the PDE residual and the boundary conditions fail; the card rejects it.
* **Harmonic field with the wrong boundary**: the residual, conservation and maximum principle all pass (it *is* a
  solution of Laplace's equation), only the boundary check on the top edge fails. Without that check this field
  would pass any "does it satisfy the PDE?" test.
* **Extrapolation**: inside $[0,1]^2$ the network is fine, but a third of the queried points lie outside the region
  it was trained on. The applicability check says so directly, and above $y = 1$ the field no longer satisfies the
  equation, the side boundary conditions or the maximum principle: the card rejects it. On a query only slightly
  outside the training region the physics checks may still pass; the applicability check alone then asks for review.

Add a reference when you have one (`reference=` an array or a function): it is a critical check too.

In [ ]:
if not SMOKE:                      # the smoke run trains too little for the control to pass
    assert cards["trained PINN"].decision == "APPROVED"
assert cards["under-trained PINN"].decision == "REJECT"
assert cards["harmonic, wrong boundary"].decision == "REJECT"
assert any(c.kind == "generalization" and c.passed is False for c in cards["trained PINN, y up to 1.5"].checks)
print("every bad prediction was caught")